#### Cleaning Exploration: Hotel Reviews

In [ ]:
import re
from datetime import datetime
import pandas as pd
import html

In [ ]:
import sys
from pathlib import Path
from config.config import RAW_INGESTED_CSV

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [22]:
df = pd.read_csv(RAW_INGESTED_CSV)
df.shape

(254573, 5)

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 254573 entries, 0 to 254572
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   city        254573 non-null  str  
 1   hotel_name  254573 non-null  str  
 2   date_raw    252986 non-null  str  
 3   title_raw   254571 non-null  str  
 4   review_raw  228784 non-null  str  
dtypes: str(5)
memory usage: 9.7 MB


In [24]:
# How many reviews per city? 
df['city'].value_counts()

city
london           79349
new-york-city    55143
san-francisco    30401
las-vegas        26506
montreal         18663
chicago          18231
dubai            11834
beijing           5253
new-delhi         4917
shanghai          4276
Name: count, dtype: int64

In [25]:
# unique hotels?
df['hotel_name'].nunique()

2567

In [26]:
## Missing values
null_counts = df.isnull().sum()
null_counts

city              0
hotel_name        0
date_raw       1587
title_raw         2
review_raw    25789
dtype: int64

In [27]:
# Inspect a sample of rows with missing date_raw 
df[df['date_raw'].isnull()].sample(min(10, df['date_raw'].isnull().sum()))

,city,hotel_name,date_raw,title_raw,review_raw
136345,london,uk_england_london_thistle_westminster,NaN,Love the thistle!,I recently visited the Thistle Westminster as ...
13895,chicago,usa_illinois_chicago_hotel_monaco_chicago_a_ki...,NaN,Perfection,Our family of 3 stayed for 4 nights over the l...
138523,london,uk_england_london_travelodge_london_southwark,NaN,CLEAN GOOD LOCATION CHEAP,"Stayed in London travelodges before, but never..."
49168,las-vegas,usa_nevada_las-vegas_mirage_hotel_casino,NaN,All right but there are better in vegas,We had reserved a King non smoking room with s...
26765,dubai,are_dubai_golden_sands_hotel_apartments,NaN,Clean spacious and cheap for Dubai - but avoid...,We stayed at Golden Sands 3 (make sure you kno...
169486,new-york-city,usa_new york city_bentley_hotel,NaN,Good price/Poor location,I stayed with my partner for 4 nights at this ...
223440,san-francisco,usa_san francisco_chancellor_hotel_on_union_sq...,NaN,Superb...everything about it!!,This was actually my boyfriend's favourite hot...
60620,las-vegas,usa_nevada_las-vegas_venetian_resort_hotel_casino,NaN,Would have been good but it never got beyond b...,The way this hotel has tried to capture all th...
13231,chicago,usa_illinois_chicago_hotel_burnham_a_kimpton_h...,NaN,Excellent stay at a historic landmark!,Stayed in the Burnham for 3 nights in Septembe...
69275,london,uk_england_london_brown_s_hotel,NaN,Great hotel let down by poor service,"This a great hotel in the heart of Mayfair, we..."


In [28]:
# Inspect rows with missing title_raw 
df[df['title_raw'].isnull()]

,city,hotel_name,date_raw,title_raw,review_raw
62839,london,uk_england_london_alexandra_hotel,Jul 15 2007,NaN,NaN
98659,london,uk_england_london_lords_hotel,Sep 30 2009,NaN,"the room was too small for 4 persons, small ba..."


In [29]:
# checking duplicates
print(df.duplicated().sum())

4


In [30]:
# Date format variety
df['date_raw'].dropna().sample(30).tolist()

['Jun 21 2004',
 'Jul 24 2007',
 'Apr 29 2005',
 'Aug 5 2008',
 'Nov 19 2009',
 'Aug 26 2008',
 'Apr 17 2006',
 'Aug 10 2009',
 'May 7 2008',
 'Nov 23 2005',
 'Sep 10 2009',
 'Aug 18 2009',
 'Jun 2 2008',
 'Nov 22 2006',
 'Sep 28 2009',
 'May 23 2007',
 'Jun 1 2009',
 'Jul 7 2006',
 'Oct 30 2005',
 'Nov 18 2009',
 'Jun 29 2006',
 'Jan 1 2008',
 'Dec 22 2007',
 'Nov 1 2009',
 'Jun 18 2007',
 'Oct 12 2009',
 'Feb 16 2007',
 'Dec 28 2008',
 'Jun 11 2004',
 'May 22 2009']

In [31]:
## Try parsing dates and see what fails

def try_parse_date(date_str):
    if pd.isnull(date_str):
        return None
    date_str = date_str.strip()
    formats_to_try = [
        "%b %d %Y",   
        "%B %d %Y",   
        "%b %d, %Y",  
        "%B %d, %Y",  
    ]
    for fmt in formats_to_try:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None  

In [32]:
sample_dates = df['date_raw'].dropna().sample(2000, random_state=42)
parsed = sample_dates.apply(try_parse_date)
failed = sample_dates[parsed.isnull()]

print(f"Sampled: {len(sample_dates)}")
print(f"Failed to parse: {len(failed)}")
failed.head(20).tolist()

Sampled: 2000
Failed to parse: 0


[]

In [37]:
# Review text length distribution
review_lengths = df['review_raw'].dropna().str.len()
review_lengths.describe()

count    228784.000000
mean       1001.785252
std         797.761506
min           2.000000
25%         495.000000
50%         795.000000
75%        1258.000000
max       19875.000000
Name: review_raw, dtype: float64

In [40]:
review_lengths[:5]

0    1565
1    1152
2    1686
3    1919
4     506
Name: review_raw, dtype: int64

In [43]:
# Shortest non-null reviews - check for junk / whitespace-only entries
df.loc[review_lengths.sort_values().index[:20], ['city', 'hotel_name', 'title_raw', 'review_raw']]

,city,hotel_name,title_raw,review_raw
3994,beijing,china_beijing_renaissance_beijing_hotel,Loved,fd
174957,new-york-city,usa_new york city_courtyard_new_york_manhattan...,cool hotel,i like it!
84462,london,uk_england_london_hilton_london_kensington,simply loved it,Great hotel
95942,london,uk_england_london_leisure_inn,good weekend,"clean,cheap"
163480,new-delhi,india_new delhi_taj_palace_hotel,Great service,Great service
132838,london,uk_england_london_the_soho_hotel,loved,beautiful hotel
179368,new-york-city,usa_new york city_flatotel,Terrible,Its says it all
162068,new-delhi,india_new delhi_hyatt_regency_delhi,nice service,nice place to be
162043,new-delhi,india_new delhi_hyatt_regency_delhi,Wonderful service,Excellent service
9006,chicago,usa_illinois_chicago_crowne_plaza_chicago_the_...,This Hotel Is Great,I love this hotel.


In [44]:
## Look for leftover encoding artifacts
has_replacement_char = df['review_raw'].dropna().str.contains('\ufffd')
print(f"Reviews containing replacement character: {has_replacement_char.sum()}")
df.loc[has_replacement_char[has_replacement_char].index[:5], ['city', 'hotel_name', 'review_raw']]

Reviews containing replacement character: 9465


,city,hotel_name,review_raw
108,beijing,china_beijing_beijing_dong_fang_hotel,This was the first hotel we stayed in as part ...
110,beijing,china_beijing_beijing_dong_fang_hotel,this is a comfortable and clean 4 star hotel c...
204,beijing,china_beijing_beijing_hotel,We were supposedly staying at the Gran Hotel B...
241,beijing,china_beijing_beijing_international_hotel,A group of us stayed here as a launching off p...
255,beijing,china_beijing_beijing_international_hotel,As part of a student group we had many expacta...


In [45]:
# Common HTML entities that need decoding (e.g. &amp; &quot;)
has_html_entity = df['review_raw'].dropna().str.contains(r'&[a-zA-Z]+;')
print(f"Reviews containing HTML entities: {has_html_entity.sum()}")
df.loc[has_html_entity[has_html_entity].index[:5], ['city', 'hotel_name', 'review_raw']]

Reviews containing HTML entities: 50213


,city,hotel_name,review_raw
16,beijing,china_beijing_ascott_beijing,Stay here for 5 nights with my family after re...
23,beijing,china_beijing_ascott_beijing,My friends and I stayed at this apartment hote...
29,beijing,china_beijing_ascott_beijing,I wasn't looking for an apartment - just somew...
31,beijing,china_beijing_ascott_beijing,The Ascott Beijing is perfect for those wantin...
34,beijing,china_beijing_ascott_beijing,My family recently returned from a 10 day tour...


In [ ]:
#  Prototype cleaning functions 
def clean_review_text(text):
    if pd.isnull(text):
        return None
    text = html.unescape(text)          
    text = text.replace('\ufffd', '')   
    text = re.sub(r'\s+', ' ', text)    
    return text.strip()


def parse_date(date_str):
    return try_parse_date(date_str)


# cleaning functions on a small sample
sample = df.sample(10, random_state=1).copy()
sample['review_clean'] = sample['review_raw'].apply(clean_review_text)
sample['date_parsed'] = sample['date_raw'].apply(parse_date)
sample[['date_raw', 'date_parsed', 'review_raw', 'review_clean']]

,date_raw,date_parsed,review_raw,review_clean
208798,May 26 2009,2009-05-26,I guess the title says it all.. It was our sec...,I guess the title says it all.. It was our sec...
86281,Aug 11 2009,2009-08-11,"This hotel costed us less, which I think is du...","This hotel costed us less, which I think is du..."
254202,Jun 6 2009,2009-06-06,My Chinese partner and I did checked in in thi...,My Chinese partner and I did checked in in thi...
32149,Jun 16 2009,2009-06-16,There was a big group of us enroute back to th...,There was a big group of us enroute back to th...
113034,Dec 4 2006,2006-12-04,We ended up here by mistake - booked one of la...,We ended up here by mistake - booked one of la...
22693,Nov 17 2008,2008-11-17,I could not recommend this Hotel more highly. ...,I could not recommend this Hotel more highly. ...
22191,Nov 18 2005,2005-11-18,I travel to Chicago quite often for business a...,I travel to Chicago quite often for business a...
25782,Sep 8 2008,2008-09-08,I stayed at the Crowne Plaza Festival City Hot...,I stayed at the Crowne Plaza Festival City Hot...
97362,Oct 17 2009,2009-10-17,"I managed to get a deal for a Sunday night, th...","I managed to get a deal for a Sunday night, th..."
19231,May 11 2009,2009-05-11,"For a four or five star hotel, was not overly ...","For a four or five star hotel, was not overly ..."
